# Case Study for Practical Machine Learning: Detecting Cracks in Images.

## Scenarios description:

> In this scenario, machine learning can be applied to detect abnormal cracks in images captured from a factory's equipment or infrastructure. The goal is to automatically identify potential defects or damages that could lead to failures, ensuring the equipment's safety and operational efficiency.

> By using machine learning and deep models, the system learns to differentiate between normal wear and tear and more significant cracks that need attention. This can help in predictive maintenance, reducing downtime and preventing costly repairs.

![](https://drive.google.com/uc?export=view&id=15DykdcKWnw5ZmhPJSgKh8YqL-Kuy1e-I)


## In this case study you will

1. Recall machine learning algorithms, deep learning knowledge that can be applied to solve this problem

2. Apply these algorithms to build several models that are capable of predicting whether an image contains cracks.

3. Practice the basic pipeline for deep learning task:

> a. Define the dataset (load the data, dataset splitting) \
  b. Define the model, including model structure and forward function. \
  c. Define the optimizer and loss function. \
  d. Set other parameters (training epochs) and start training.

In [1]:
"""
Step 1: import crucial packages
"""
import os
import numpy as np
import cv2  # OpenCV for image processing
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

In [2]:
"""
Step 2: get the dataset.
- Google Colab: upload Concrete_Crack_Images.zip when prompted.
- Running locally: put Concrete_Crack_Images.zip in the repo's data/ folder (see data/README.md).
"""
import os, zipfile

ZIP_NAME = 'Concrete_Crack_Images.zip'
try:
    from google.colab import files  # only available on Colab
    if not os.path.exists(ZIP_NAME):
        files.upload()
    zip_path = ZIP_NAME
except ImportError:
    zip_path = os.path.join('..', 'data', ZIP_NAME)

if not os.path.isdir('Concrete_Crack_Images'):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('Concrete_Crack_Images')
print(os.listdir('Concrete_Crack_Images/Concrete_Crack_Images'))

['Negative_img', 'Positive_img']


In [3]:
"""
Step 3: load the datasets and make dataset splitting
"""
# Function to load images from a directory
def load_images_from_folder(folder, label):
    images = []
    labels = []
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # Read image as grayscale
        if img is not None:
            img = cv2.resize(img, (30, 30))  # Resize to 30x30
            images.append(img.flatten())  # Flatten the image to a vector
            labels.append(label)  # Append the corresponding label
    return images, labels

# Load positive and negative images
positive_folder = './Concrete_Crack_Images/Concrete_Crack_Images/Positive_img'
negative_folder = './Concrete_Crack_Images/Concrete_Crack_Images/Negative_img'

### Load positive and negative images
pos_images, pos_labels = load_images_from_folder(positive_folder, 1)  # 1 = crack
neg_images, neg_labels = load_images_from_folder(negative_folder, 0)  # 0 = no crack

# Combine the data together
X = np.array(pos_images + neg_images)
y = np.array(pos_labels + neg_labels)
print(f"Data shape: {X.shape}, Labels shape: {y.shape}")

# Split the dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} images, Test: {X_test.shape[0]} images")

Data shape: (10000, 900), Labels shape: (10000,)
Train: 8000 images, Test: 2000 images


In [4]:
"""
Step 3: Run SVM algorithm
"""
# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # learn mean/std from training data only
X_test_scaled = scaler.transform(X_test)        # apply the same scaling to test data

# Initialize the SVM classifier
# we use 'poly' kernel
svm = SVC(kernel='poly')

# Train the SVM classifier
svm.fit(X_train_scaled, y_train)

# Make predictions on the test set
y_pred = svm.predict(X_test_scaled)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')

Accuracy: 86.65%


In [5]:
"""
Step 4: You can modify SVM's kernel type for better results
"""
# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize the SVM classifier
# Replace 'poly' kernel into other kernels
svm = SVC(kernel='rbf')  # RBF (Gaussian) kernel handles non-linear patterns better

# Train the SVM classifier
svm.fit(X_train_scaled, y_train)

# Make predictions on the test set
y_pred = svm.predict(X_test_scaled)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')

Accuracy: 96.75%
